# ДЗ 14. NumPy: векторизация, broadcasting, оси, маски 🔢
Решите задачи в Google Colab и сдайте через GitHub: папка `hw14` в репозитории `cyberformula3-homework`.

- **[База]** — обязательно для всех ·
- **[Челлендж]** — сколько получится · 
- **[Про]** — для тех, кому легко · 

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы и маски. Цикл можно там, где это сказано.

**Проверь себя перед сдачей:**
□ **Среда выполнения → Перезапустить и выполнить все** — ноутбук проходит без ошибок
□ Ответы совпадают с «Примером проверки», где он есть

In [1]:
import numpy as np
# проверка: вставьте сюда строки 1–5 и запустите

## Загрузка данных — запустите как есть

In [3]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders.csv"
print("Работаем с файлом:", FILE)

Работаем с файлом: orders.csv


In [4]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [7]:
# TODO
new_revenue = revenue * 1.05
revenue_growth = new_revenue.sum() - revenue.sum()

print(f"Выручка вырастет на: {revenue_growth:,.0f} ₸  столько магазин дополнительно заработает при повышении цен на 5%, если спрос не изменится.")

Выручка вырастет на: 4,625,325 ₸  столько магазин дополнительно заработает при повышении цен на 5%, если спрос не изменится.


### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [9]:
# TODO
smartphones_mask = (category == "Смартфоны")
smartphones_count = smartphones_mask.sum()
smartphones_revenue = revenue[smartphones_mask].sum()
smartphones_share = smartphones_mask.mean() * 100

print(f"Заказов смартфонов: {smartphones_count} на сумму {smartphones_revenue:,} ₸ (доля {smartphones_share:.1f}%).")
print("Вывод: смартфоны составляют больше четверти всех заказов.")

Заказов смартфонов: 131 на сумму 30,689,000 ₸ (доля 26.2%).
Вывод: смартфоны составляют больше четверти всех заказов.


### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [10]:
# TODO
shymkent_paid = (status == "оплачен") & (city == "Шымкент")
shymkent_count = shymkent_paid.sum()
shymkent_revenue = revenue[shymkent_paid].sum()

print(f"Оплаченных заказов из Шымкента: {shymkent_count} на сумму {shymkent_revenue:,} ₸.")
print("Вывод: это реальная подтверждённая выручка магазина по Шымкенту без учёта отмен.")

Оплаченных заказов из Шымкента: 71 на сумму 12,852,500 ₸.
Вывод: это реальная подтверждённая выручка магазина по Шымкенту без учёта отмен.


### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [12]:
w = revenue[:30].reshape(6, 5)
# TODO
# Сумма за каждый день: axis=1 (складываем вдоль строк), получится 6 чисел
daily_sums = w.sum(axis=1)

# День с самой большой суммой (индекс + 1)
best_day = daily_sums.argmax() + 1
best_day_sum = daily_sums.max()

#Самый дорогой заказ каждого дня
daily_max = w.max(axis=1)

print("1. Суммы по дням (axis=1, 6 чисел):", daily_sums)
print(f"2. Лучший день: {best_day}-й ({best_day_sum:,} ₸) пиковая выручка за этот период.")
print("3. Самые дорогие заказы по дням:", daily_max)

1. Суммы по дням (axis=1, 6 чисел): [ 699000  925000 1094000  636000 1047000  571000]
2. Лучший день: 3-й (1,094,000 ₸) пиковая выручка за этот период.
3. Самые дорогие заказы по дням: [198000 319000 549000 449000 449000 319000]


## [Челлендж]

### Задача 5. Заказы в трёх валютах
Покупатели из-за рубежа просят показать суммы заказов в своей валюте.
Курсы Нацбанка РК на 19 ноября 2026: 1 USD = 444.16 ₸, 1 EUR = 487.29 ₸, 1 RUB = 5.40 ₸.

1. Возьмите суммы **первых 5 заказов** из `revenue` и получите **одной операцией** таблицу «заказ × валюта» формы `(5, 3)`, округлите до 2 знаков.
   Какую форму нужно дать суммам заказов, чтобы сработал broadcasting?
2. Выведите рядом товар каждого из этих 5 заказов — массив `product` уже собран в ячейке загрузки.


**Пример проверки:** форма суммам — `(5, 1)`; первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — `[445.79  406.33  36666.67]`.

In [14]:
# TODO
rates = np.array([444.16, 487.29, 5.40])  # USD, EUR, RUB
top5_revenue = revenue[:5].reshape(5, 1)
in_currencies = np.round(top5_revenue / rates, 2)

print("Форма массива сумм:", top5_revenue.shape)
print("Таблица (USD, EUR, RUB) и товары:")
for name, kzt, row in zip(product[:5], revenue[:5], in_currencies):
    print(f"{name} ({kzt:,} ₸) -> {row}")

print("Вывод: цены автоматически пересчитаны в три валюты.")

Форма массива сумм: (5, 1)
Таблица (USD, EUR, RUB) и товары:
Xiaomi Redmi Note 13 (198,000 ₸) -> [  445.79   406.33 36666.67]
Клавиатура Logitech K120 (24,000 ₸) -> [  54.03   49.25 4444.44]
Samsung Galaxy Tab A9 (159,000 ₸) -> [  357.98   326.29 29444.44]
Samsung Galaxy A55 (189,000 ₸) -> [  425.52   387.86 35000.  ]
Монитор LG 27 дюймов (129,000 ₸) -> [  290.44   264.73 23888.89]
Вывод: цены автоматически пересчитаны в три валюты.


### Задача 6. Категории
Для **каждой** категории из `np.unique(category)` выведите число заказов, выручку и средний чек (цикл по категориям можно).
Какая категория приносит больше всего денег? А у какой самый большой средний чек?

**Пример проверки:** и по выручке, и по среднему чеку лидируют **Ноутбуки** — `40 992 000 ₸`, средний чек ≈ `422 598 ₸`.

In [21]:
# TODO
best_rev_cat, best_rev = "", 0
best_avg_cat, best_avg = "", 0

for cat in np.unique(category):
    mask = (category == cat)
    count = mask.sum()
    cat_rev = revenue[mask].sum()
    avg_check = revenue[mask].mean()
    print(f"{cat}: {count} заказов | выручка {cat_rev:,} ₸ | средний чек {avg_check:,.0f} ₸")
    
    if cat_rev > best_rev:
        best_rev, best_rev_cat = cat_rev, cat
    if avg_check > best_avg:
        best_avg, best_avg_cat = avg_check, cat

print(f"\nЛидер по выручке: {best_rev_cat} ({best_rev:,} ₸).")
print(f"Лидер по среднему чеку: {best_avg_cat} ({best_avg:,.0f} ₸) эта категория приносит магазину основной доход и крупные чеки.")

Аксессуары: 171 заказов | выручка 4,514,500 ₸ | средний чек 26,401 ₸
Мониторы: 50 заказов | выручка 6,046,000 ₸ | средний чек 120,920 ₸
Ноутбуки: 97 заказов | выручка 40,992,000 ₸ | средний чек 422,598 ₸
Планшеты: 51 заказов | выручка 10,265,000 ₸ | средний чек 201,275 ₸
Смартфоны: 131 заказов | выручка 30,689,000 ₸ | средний чек 234,267 ₸

Лидер по выручке: Ноутбуки (40,992,000 ₸).
Лидер по среднему чеку: Ноутбуки (422,598 ₸) эта категория приносит магазину основной доход и крупные чеки.


### Задача 7. Бесплатная доставка
Доставка бесплатна, если сумма заказа от 50 000 ₸. С помощью `np.where` подпишите каждый заказ: `"бесплатно"` или `"платно"`.
Сколько заказов получили бесплатную доставку?

**Пример проверки:** `354`.

In [20]:
# TODO
delivery = np.where(revenue >= 50000, "бесплатно", "платно")
free_count = (delivery == "бесплатно").sum()

print(f"Бесплатную доставку получили: {free_count} заказов!")

Бесплатную доставку получили: 354 заказов!


## [Про]

### Задача 8. Таблица «категория × месяц»
Постройте таблицу **числа заказов** формы `(5, 8)`: строки — категории (`np.unique(category)`), столбцы — месяцы (`np.unique(month)`).
Двойной цикл по категориям и месяцам можно, внутри — маска из двух условий.

1. Проверьте: сумма всей таблицы должна быть равна числу заказов.
2. В каком месяце было больше всего заказов смартфонов?

**Пример проверки:** сумма таблицы `500`; больше всего смартфонов — в **сентябре (9)**, `22` заказа.

In [ ]:
# TODO

### Задача 9. Доли категорий по месяцам
Разделите таблицу на итог по **каждому месяцу** — получится доля каждой категории внутри месяца.
Это broadcasting: таблица `(5, 8)` делится на строку `(8,)`. Какая `axis` даёт итог по месяцам?

1. Проверьте: сумма каждого столбца после деления равна `1`.
2. Какая доля заказов в **августе** пришлась на аксессуары?
3. Сохраните доли в `shares.json`: `{"категории": [...], "месяцы": [...], "доли": [[...], ...]}`, доли округлите до 2 знаков. Не забудьте `.tolist()`.

**Пример проверки:** в августе аксессуары — `40.4%` заказов.

In [ ]:
# TODO